In [ ]:
# Example 1: dispatch_command — the simplest argh entrypoint.
import argh
import io

def greet(name: str = "world", greeting: str = "Hello"):
    return f"{greeting}, {name}!"

buf = io.StringIO()
argh.dispatch_command(greet, argv=["--name", "argh", "--greeting", "Hi"], output_file=buf)
output = buf.getvalue().strip()
print("dispatch_command output:", repr(output))
assert output == "Hi, argh!", f"unexpected output: {output!r}"


In [ ]:
# Example 2: dispatch_commands — multiple subcommands from a flat function list.
import argh
import io

def add(a: int, b: int):
    return a + b

def mul(a: int, b: int):
    return a * b

buf = io.StringIO()
argh.dispatch_commands([add, mul], argv=["add", "3", "4"], output_file=buf)
result_add = buf.getvalue().strip()
print("add 3 4 ->", result_add)

buf2 = io.StringIO()
argh.dispatch_commands([add, mul], argv=["mul", "6", "7"], output_file=buf2)
result_mul = buf2.getvalue().strip()
print("mul 6 7 ->", result_mul)

assert result_add == "7"
assert result_mul == "42"


In [ ]:
# Example 3: ArghParser + set_default_command — manual parser composition.
import argh
import io

def echo(text: str, upper: bool = False):
    return text.upper() if upper else text

parser = argh.ArghParser()
argh.set_default_command(parser, echo)

buf = io.StringIO()
argh.dispatch(parser, argv=["hello", "--upper"], output_file=buf)
out = buf.getvalue().strip()
print("ArghParser dispatch output:", out)
assert out == "HELLO"


In [ ]:
# Example 4: @arg decorator + add_commands for subcommand groups.
import argh
import io

@argh.arg("--loud", help="shout the result")
def shout(text: str, loud: bool = False):
    return text.upper() + "!" if loud else text

def quiet(text: str):
    return text.lower()

parser = argh.ArghParser()
argh.add_commands(parser, [shout, quiet])

buf = io.StringIO()
argh.dispatch(parser, argv=["shout", "hello", "--loud"], output_file=buf)
out = buf.getvalue().strip()
print("shout --loud output:", out)
assert out == "HELLO!"


In [ ]:
# Example 5: generator results stream as lines; aliases group subcommands.
import argh
import io

def counts(n: int = 3):
    for i in range(n):
        yield f"line-{i}"

parser = argh.ArghParser()
argh.add_commands(parser, [counts], group_name="util", group_kwargs={"help": "utility cmds"})

buf = io.StringIO()
argh.dispatch(parser, argv=["util", "counts", "-n", "4"], output_file=buf)
lines = [ln for ln in buf.getvalue().splitlines() if ln]
print("generator lines:", lines)
assert lines == ["line-0", "line-1", "line-2", "line-3"]
